# NVIDIA Parakeet — Streaming STT with Latency Benchmarks

This notebook runs **nvidia/parakeet-rnnt-1.1b** (FastConformer Transducer, 1.1B params) in a **chunked streaming** mode, simulating real-time audio input.

**What it does:**
- Installs NeMo + dependencies
- Downloads the Parakeet model
- Runs streaming inference on chunks of audio (configurable chunk size)
- Measures per-chunk and end-to-end latency
- Compares batch vs streaming transcription quality

> **GPU required.** This notebook is designed for NVIDIA GPU environments (Talus, Colab, etc.).

---

## 1. Install Dependencies

NeMo has complex dependencies. This cell handles the full install chain.
If you already have NeMo installed, skip this cell.

In [ ]:
%%capture install_log
# ── System packages (Debian/Ubuntu) ──────────────────────────────────────────
import subprocess, sys, os

# Install system audio libs if running on Linux (Colab / Talus / Docker)
if sys.platform == 'linux':
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    subprocess.run(
        ['apt-get', 'install', '-y', '-qq',
         'sox', 'libsndfile1', 'ffmpeg', 'portaudio19-dev'],
        check=False,
    )

# ── Python packages ──────────────────────────────────────────────────────────
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'nemo_toolkit[asr]',
    'librosa',
    'soundfile',
    'matplotlib',
    'ipywidgets',
    'pyaudio',
], check=True)

print('Installation complete.')

In [ ]:
# Show install log if you hit issues
# print(install_log.stdout)

## 2. Verify Environment

In [ ]:
import torch
import numpy as np
import time
import os

print(f'PyTorch    : {torch.__version__}')
print(f'CUDA avail : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU Memory : {gpu_mem:.1f} GB')
else:
    print('WARNING: No GPU detected. Parakeet 1.1B will be very slow on CPU.')

import nemo
import nemo.collections.asr as nemo_asr
print(f'NeMo       : {nemo.__version__}')

## 3. Load Model

We use `parakeet-rnnt-1.1b` (RNNT decoder — natively supports streaming inference).

Alternatives you can swap in:
- `nvidia/parakeet-tdt-1.1b` — 64% faster, slightly better accuracy
- `nvidia/parakeet-ctc-1.1b` — CTC decoder, simplest but no native streaming
- `nvidia/parakeet-rnnt-0.6b` — smaller, faster on limited GPU memory

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
MODEL_NAME = "nvidia/parakeet-rnnt-1.1b"   # Change this to try other variants
SAMPLE_RATE = 16000                         # Parakeet expects 16 kHz mono

# ── Load ─────────────────────────────────────────────────────────────────────
print(f'Loading {MODEL_NAME} ...')
t0 = time.time()
asr_model = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained(model_name=MODEL_NAME)
load_time = time.time() - t0

# Move to GPU and set eval mode
device = 'cuda' if torch.cuda.is_available() else 'cpu'
asr_model = asr_model.to(device)
asr_model.eval()
asr_model.freeze()  # Disable gradient computation entirely

print(f'Model loaded in {load_time:.1f}s on {device}')
print(f'Parameters : {sum(p.numel() for p in asr_model.parameters()) / 1e6:.0f}M')

if torch.cuda.is_available():
    mem_alloc = torch.cuda.memory_allocated() / 1e9
    print(f'GPU memory used: {mem_alloc:.2f} GB')

## 4. Download Sample Audio

We need a test audio file. If you have your own, set `AUDIO_PATH` to its path and skip the download.

In [ ]:
import urllib.request
import librosa
import soundfile as sf

# ── Download a sample WAV if not present ─────────────────────────────────────
AUDIO_URL = 'https://dldata-public.s3.us-east-2.amazonaws.com/2086-149220-0033.wav'
AUDIO_PATH = 'sample_audio.wav'

if not os.path.exists(AUDIO_PATH):
    print(f'Downloading sample audio...')
    urllib.request.urlretrieve(AUDIO_URL, AUDIO_PATH)
    print('Done.')

# ── Load and validate ────────────────────────────────────────────────────────
audio, sr = librosa.load(AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
duration = len(audio) / SAMPLE_RATE

print(f'Audio file    : {AUDIO_PATH}')
print(f'Sample rate   : {sr} Hz')
print(f'Duration      : {duration:.2f}s')
print(f'Samples       : {len(audio):,}')
print(f'Dtype         : {audio.dtype}')
print(f'Range         : [{audio.min():.3f}, {audio.max():.3f}]')

## 5. Baseline — Full-File Transcription (Non-Streaming)

First, let's get a **baseline** by transcribing the full audio at once. This gives us:
- Ground truth for quality comparison
- Baseline latency number

In [ ]:
# ── CUDA warmup (first inference is always slower due to kernel compilation) ─
if device == 'cuda':
    print('Warming up CUDA kernels...')
    warmup_audio = np.zeros(SAMPLE_RATE, dtype=np.float32)  # 1s silence
    sf.write('/tmp/_warmup.wav', warmup_audio, SAMPLE_RATE)
    _ = asr_model.transcribe(['/tmp/_warmup.wav'])
    torch.cuda.synchronize()
    print('Warmup done.\n')

# ── Full-file transcription ──────────────────────────────────────────────────
torch.cuda.synchronize() if device == 'cuda' else None
t0 = time.perf_counter()
output = asr_model.transcribe([AUDIO_PATH])
torch.cuda.synchronize() if device == 'cuda' else None
t1 = time.perf_counter()

baseline_text = output[0].text if hasattr(output[0], 'text') else output[0]
baseline_latency_ms = (t1 - t0) * 1000
rtf = (t1 - t0) / duration

print(f'Baseline Transcription:')
print(f'  "{baseline_text}"')
print(f'\nLatency : {baseline_latency_ms:.0f} ms')
print(f'RTF     : {rtf:.4f}  (Real-Time Factor, <1.0 = faster than real-time)')
print(f'RTFx    : {1/rtf:.0f}x  (times faster than real-time)')

## 6. Streaming Inference — Chunked Transcription

This simulates a real-time streaming scenario:
1. Audio is split into small chunks (configurable: 0.5s, 1s, 2s, etc.)
2. Each chunk is written to a temp file and transcribed
3. Per-chunk latency is measured
4. Results are accumulated to form the full transcript

### Why chunk-based?
Parakeet RNNT is a transducer model — it naturally processes audio frame-by-frame.
By feeding it small chunks, we simulate how a real streaming pipeline would work.

> **Key trade-off:** Smaller chunks → lower latency but potentially lower accuracy
> at chunk boundaries. Larger chunks → higher accuracy but higher latency.

In [ ]:
import tempfile

def streaming_transcribe(
    audio_array: np.ndarray,
    model,
    sample_rate: int = 16000,
    chunk_duration_s: float = 2.0,
    overlap_duration_s: float = 0.5,
    device: str = 'cuda',
    verbose: bool = True,
):
    """
    Simulate streaming transcription by processing audio in overlapping chunks.
    """
    chunk_samples = int(chunk_duration_s * sample_rate)
    overlap_samples = int(overlap_duration_s * sample_rate)
    step_samples = chunk_samples - overlap_samples
    
    total_samples = len(audio_array)
    chunks_results = []
    all_texts = []
    latencies = []
    
    if verbose:
        n_chunks = (total_samples - overlap_samples) // step_samples + 1
        print(f'Chunk size     : {chunk_duration_s}s ({chunk_samples} samples)')
        print(f'Overlap        : {overlap_duration_s}s ({overlap_samples} samples)')
        print(f'Step           : {chunk_duration_s - overlap_duration_s}s ({step_samples} samples)')
        print(f'Expected chunks: ~{n_chunks}')
        print(f'Audio duration : {total_samples / sample_rate:.2f}s')
        print('-' * 70)
    
    total_t0 = time.perf_counter()
    chunk_idx = 0
    pos = 0
    
    while pos < total_samples:
        # Extract chunk
        end = min(pos + chunk_samples, total_samples)
        chunk = audio_array[pos:end]
        
        # Pad if last chunk is too short (< 0.1s can cause issues)
        if len(chunk) < int(0.1 * sample_rate):
            break
        
        # Write chunk to temp file (NeMo's transcribe API expects file paths)
        tmp_path = f'/tmp/_chunk_{chunk_idx}.wav'
        sf.write(tmp_path, chunk, sample_rate)
        
        # ── Transcribe with timing ───────────────────────────────────────
        if device == 'cuda':
            torch.cuda.synchronize()
        
        t0 = time.perf_counter()
        result = model.transcribe([tmp_path])
        
        if device == 'cuda':
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        
        # Extract text
        text = result[0].text if hasattr(result[0], 'text') else str(result[0])
        text = text.strip()
        
        latency_ms = (t1 - t0) * 1000
        chunk_dur = len(chunk) / sample_rate
        
        chunk_result = {
            'idx': chunk_idx,
            'start_s': pos / sample_rate,
            'end_s': end / sample_rate,
            'duration_s': chunk_dur,
            'text': text,
            'latency_ms': latency_ms,
            'rtf': (t1 - t0) / chunk_dur,
        }
        chunks_results.append(chunk_result)
        latencies.append(latency_ms)
        
        if text:
            all_texts.append(text)
        
        if verbose:
            print(f'[{chunk_idx:03d}] '
                  f'{pos/sample_rate:6.2f}s-{end/sample_rate:6.2f}s  '
                  f'latency={latency_ms:6.0f}ms  '
                  f'RTFx={1/chunk_result["rtf"]:5.0f}x  '
                  f'| {text}')
        
        # Clean up temp file
        os.remove(tmp_path)
        
        pos += step_samples
        chunk_idx += 1
    
    total_t1 = time.perf_counter()
    total_latency = (total_t1 - total_t0) * 1000
    audio_duration = total_samples / sample_rate
    
    latencies_np = np.array(latencies)
    
    summary = {
        'full_text': ' '.join(all_texts),
        'chunks': chunks_results,
        'n_chunks': len(chunks_results),
        'total_latency_ms': total_latency,
        'avg_latency_ms': float(np.mean(latencies_np)),
        'p50_latency_ms': float(np.percentile(latencies_np, 50)),
        'p95_latency_ms': float(np.percentile(latencies_np, 95)),
        'p99_latency_ms': float(np.percentile(latencies_np, 99)),
        'min_latency_ms': float(np.min(latencies_np)),
        'max_latency_ms': float(np.max(latencies_np)),
        'rtf': total_latency / 1000 / audio_duration,
        'audio_duration_s': audio_duration,
    }
    
    return summary

### 6a. Run Streaming with 2s Chunks

In [ ]:
result_2s = streaming_transcribe(
    audio, asr_model,
    sample_rate=SAMPLE_RATE,
    chunk_duration_s=2.0,
    overlap_duration_s=0.5,
    device=device,
    verbose=True,
)

print('\n' + '=' * 70)
print(f'STREAMING RESULT (2s chunks):')
print(f'  "{result_2s["full_text"]}"')
print(f'\nBASELINE:')
print(f'  "{baseline_text}"')
print(f'\nLATENCY STATS:')
print(f'  avg    = {result_2s["avg_latency_ms"]:.0f} ms')
print(f'  p50    = {result_2s["p50_latency_ms"]:.0f} ms')
print(f'  p95    = {result_2s["p95_latency_ms"]:.0f} ms')
print(f'  min    = {result_2s["min_latency_ms"]:.0f} ms')
print(f'  max    = {result_2s["max_latency_ms"]:.0f} ms')
print(f'  total  = {result_2s["total_latency_ms"]:.0f} ms')
print(f'  RTFx   = {1/result_2s["rtf"]:.0f}x faster than real-time')
print('=' * 70)

### 6b. Run Streaming with 1s Chunks (Lower Latency)

In [ ]:
result_1s = streaming_transcribe(
    audio, asr_model,
    sample_rate=SAMPLE_RATE,
    chunk_duration_s=1.0,
    overlap_duration_s=0.25,
    device=device,
    verbose=True,
)

print('\n' + '=' * 70)
print(f'STREAMING RESULT (1s chunks):')
print(f'  "{result_1s["full_text"]}"')
print(f'\nLATENCY STATS:')
print(f'  avg    = {result_1s["avg_latency_ms"]:.0f} ms')
print(f'  p50    = {result_1s["p50_latency_ms"]:.0f} ms')
print(f'  p95    = {result_1s["p95_latency_ms"]:.0f} ms')
print(f'  min    = {result_1s["min_latency_ms"]:.0f} ms')
print(f'  max    = {result_1s["max_latency_ms"]:.0f} ms')
print(f'  RTFx   = {1/result_1s["rtf"]:.0f}x faster than real-time')
print('=' * 70)

## 7. Latency Comparison — Multiple Chunk Sizes

Benchmark across several chunk sizes to find the sweet spot.

In [ ]:
chunk_sizes = [0.5, 1.0, 2.0, 3.0, 5.0]
benchmark_results = {}

print(f'{"Chunk":>6s} {"Avg(ms)":>8s} {"P50(ms)":>8s} {"P95(ms)":>8s} '
      f'{"Min(ms)":>8s} {"Max(ms)":>8s} {"RTFx":>6s} {"Chunks":>7s}')
print('-' * 72)

for cs in chunk_sizes:
    overlap = min(0.5, cs * 0.25)  # 25% overlap, capped at 0.5s
    r = streaming_transcribe(
        audio, asr_model,
        sample_rate=SAMPLE_RATE,
        chunk_duration_s=cs,
        overlap_duration_s=overlap,
        device=device,
        verbose=False,
    )
    benchmark_results[cs] = r
    print(f'{cs:5.1f}s {r["avg_latency_ms"]:8.0f} {r["p50_latency_ms"]:8.0f} '
          f'{r["p95_latency_ms"]:8.0f} {r["min_latency_ms"]:8.0f} '
          f'{r["max_latency_ms"]:8.0f} {1/r["rtf"]:5.0f}x {r["n_chunks"]:7d}')

## 8. Latency Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Average latency vs chunk size ─────────────────────────────────
cs_list = sorted(benchmark_results.keys())
avg_lat = [benchmark_results[cs]['avg_latency_ms'] for cs in cs_list]
p95_lat = [benchmark_results[cs]['p95_latency_ms'] for cs in cs_list]

axes[0].plot(cs_list, avg_lat, 'o-', label='Avg', linewidth=2, markersize=8)
axes[0].plot(cs_list, p95_lat, 's--', label='P95', linewidth=2, markersize=8)
axes[0].set_xlabel('Chunk Size (seconds)', fontsize=12)
axes[0].set_ylabel('Latency (ms)', fontsize=12)
axes[0].set_title('Latency vs Chunk Size', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# ── Plot 2: Per-chunk latency distribution for 2s chunks ─────────────────
chunk_latencies = [c['latency_ms'] for c in result_2s['chunks']]
chunk_indices = list(range(len(chunk_latencies)))
axes[1].bar(chunk_indices, chunk_latencies, color='steelblue', alpha=0.8)
axes[1].axhline(y=result_2s['avg_latency_ms'], color='red', linestyle='--',
                label=f'Avg={result_2s["avg_latency_ms"]:.0f}ms')
axes[1].set_xlabel('Chunk Index', fontsize=12)
axes[1].set_ylabel('Latency (ms)', fontsize=12)
axes[1].set_title('Per-Chunk Latency (2s chunks)', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('parakeet_latency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: parakeet_latency.png')

## 8b. Real-Time Microphone Streaming

This uses your microphone for real-time inference using the pure-Python `sounddevice` library instead of PyAudio avoid compilation issues on Linux.

In [ ]:
import pyaudio as pa
import time
import numpy as np
import torch

# ── Configuration ──────────────────────────────────────────────────────
CHUNK_SEC = 1.0
OVERLAP_SEC = 0.2

p = pa.PyAudio()
print('Available audio input devices:')
for i in range(p.get_device_count()):
    dev = p.get_device_info_by_index(i)
    if dev.get('maxInputChannels') > 0:
        print(f"{i}: {dev.get('name')}")
print('
---')
dev_idx_str = input('Please type input device ID (leave blank for default): ')
dev_idx = int(dev_idx_str) if dev_idx_str.strip() else None

# ── Setup Frame Buffer ───────────────────────────────────────────
class FrameBuffer:
    def __init__(self, sr, chunk_sec, overlap_sec):
        self.sr = sr
        self.chunk_len = int(chunk_sec * sr)
        self.overlap_len = int(overlap_sec * sr)
        self.buffer = np.zeros(self.chunk_len + self.overlap_len, dtype=np.float32)
        self.filled = 0
    def add(self, audio):
        needed = self.chunk_len + self.overlap_len - self.filled
        if len(audio) < needed:
            self.buffer[self.filled:self.filled+len(audio)] = audio
            self.filled += len(audio)
            return None
        self.buffer[self.filled:self.filled+needed] = audio[:needed]
        out = self.buffer.copy()
        self.buffer[:self.overlap_len] = self.buffer[-self.overlap_len:]
        self.filled = self.overlap_len
        return out

buf = FrameBuffer(SAMPLE_RATE, CHUNK_SEC, OVERLAP_SEC)

# Preprocessor config mapping
cfg = asr_model._cfg
cfg.preprocessor.dither = 0.0
cfg.preprocessor.pad_to = 0
cfg.preprocessor.normalize = {
    "fixed_mean": [
        -14.95827016, -12.71798736, -11.76067913, -10.83311182,
        -10.6746914, -10.15163465, -10.05378331, -9.53918999,
        -9.41858904, -9.23382904, -9.46470918, -9.56037,
        -9.57434245, -9.47498732, -9.7635205, -10.08113074,
        -10.05454561, -9.81112681, -9.68673603, -9.83652977,
        -9.90046248, -9.85404766, -9.92560366, -9.95440354,
        -10.17162966, -9.90102482, -9.47471025, -9.54416855,
        -10.07109475, -9.98249912, -9.74359465, -9.55632283,
        -9.23399915, -9.36487649, -9.81791084, -9.56799225,
        -9.70630899, -9.85148006, -9.8594418, -10.01378735,
        -9.98505315, -9.62016094, -10.342285, -10.41070709,
        -10.10687659, -10.14536695, -10.30828702, -10.23542833,
        -10.88546868, -11.31723646, -11.46087382, -11.54877829,
        -11.62400934, -11.92190509, -12.14063815, -11.65130117,
        -11.58308531, -12.22214663, -12.42927197, -12.58039805,
        -13.10098969, -13.14345864, -13.31835645, -14.47345634
    ],
    "fixed_std": [
        3.81402054, 4.12647781, 4.05007065, 3.87790987,
        3.74721178, 3.68377423, 3.69344, 3.54001005,
        3.59530412, 3.63752368, 3.62826417, 3.56488469,
        3.53740577, 3.68313898, 3.67138151, 3.55707266,
        3.54919572, 3.55721289, 3.56723346, 3.46029304,
        3.44119672, 3.49030548, 3.39328435, 3.28244406,
        3.28001423, 3.26744937, 3.46692348, 3.35378948,
        2.96330901, 2.97663111, 3.04575148, 2.89717604,
        2.95659301, 2.90181116, 2.7111687, 2.93041291,
        2.86647897, 2.73473181, 2.71495654, 2.75543763,
        2.79174615, 2.96076456, 2.57376336, 2.68789782,
        2.90930817, 2.90412004, 2.76187531, 2.89905006,
        2.65896173, 2.81032176, 2.87769857, 2.84665271,
        2.80863137, 2.80707634, 2.83752184, 3.01914511,
        2.92046439, 2.78461139, 2.90034605, 2.94599508,
        2.99099718, 3.0167554, 3.04649716, 2.94116777
    ]
}
asr_model.preprocessor = asr_model.from_config_dict(cfg.preprocessor)

# ── Audio Callback ──────────────────────────────────────────────────
def audio_callback(in_data, frame_count, time_info, status):
    # PyAudio gives us int16 bytes. We must convert to float32 [-1.0, 1.0]
    audio = np.frombuffer(in_data, dtype=np.int16).astype(np.float32) / 32768.0
    
    chunk = buf.add(audio)
    if chunk is None:
        return (in_data, pa.paContinue)
    
    torch.cuda.synchronize() if device == 'cuda' else None
    start = time.time()
    logits, _, _ = asr_model.forward(
        input_signal=torch.tensor(chunk).unsqueeze(0).to(device),
        input_signal_length=torch.tensor([len(chunk)]).to(device),
    )
    torch.cuda.synchronize() if device == 'cuda' else None
    latency = (time.time() - start) * 1000
    
    pred = asr_model.decoding.ctc_decoder_predictions_tensor(logits)
    print(f'[+{latency:.0f} ms] {pred[0]}')
    
    return (in_data, pa.paContinue)

# ── Start Stream ────────────────────────────────────────────────────
stream = p.open(
    format=pa.paInt16,
    channels=1,
    rate=SAMPLE_RATE,
    input=True,
    frames_per_buffer=int(SAMPLE_RATE * CHUNK_SEC),
    input_device_index=dev_idx,
    stream_callback=audio_callback
)

print('
Listening... (Interrupt kernel (Stop cell) to exit)')
stream.start_stream()
try:
    while stream.is_active():
        time.sleep(0.1)
except KeyboardInterrupt:
    print('
Stopped.')
finally:
    stream.stop_stream()
    stream.close()
    p.terminate()

## 9. Streaming with Your Own Audio

Upload your own WAV/MP3/FLAC file and run streaming inference.
The cell handles format conversion automatically.

In [ ]:
# ── Set your audio file path here ────────────────────────────────────────────
YOUR_AUDIO = None  # e.g. '/path/to/your/audio.wav'

if YOUR_AUDIO and os.path.exists(YOUR_AUDIO):
    # Load and resample to 16kHz mono
    your_audio, _ = librosa.load(YOUR_AUDIO, sr=SAMPLE_RATE, mono=True)
    print(f'Loaded: {YOUR_AUDIO}')
    print(f'Duration: {len(your_audio)/SAMPLE_RATE:.2f}s')
    
    # Run streaming
    your_result = streaming_transcribe(
        your_audio, asr_model,
        sample_rate=SAMPLE_RATE,
        chunk_duration_s=2.0,
        overlap_duration_s=0.5,
        device=device,
        verbose=True,
    )
    
    print(f'\nFull transcript: "{your_result["full_text"]}"')
    print(f'Avg latency: {your_result["avg_latency_ms"]:.0f}ms')
else:
    print('Set YOUR_AUDIO to a valid file path to run this cell.')
    print('Example: YOUR_AUDIO = \'my_recording.wav\'')

## 10. GPU Memory & Cleanup

In [ ]:
if torch.cuda.is_available():
    print(f'GPU memory allocated : {torch.cuda.memory_allocated() / 1e9:.2f} GB')
    print(f'GPU memory reserved  : {torch.cuda.memory_reserved() / 1e9:.2f} GB')
    print(f'GPU memory peak      : {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')

# Uncomment to free GPU memory:
# del asr_model
# torch.cuda.empty_cache()
# print('Model unloaded, GPU memory freed.')

---

## Summary

| Metric | Value |
|--------|-------|
| Model | nvidia/parakeet-rnnt-1.1b |
| Parameters | ~1.1B |
| Input | 16kHz mono audio |
| Streaming | Chunked (configurable 0.5s–5s) |
| Overlap | Configurable (prevents boundary artifacts) |

### Key Findings
- **Smaller chunks** → lower per-chunk latency but more chunks, potential boundary issues
- **Larger chunks** → higher per-chunk latency but better transcription quality
- **Sweet spot** is typically **1–2s chunks** with 0.25–0.5s overlap
- First inference is always slower (CUDA kernel compilation) — always warmup!

### Blindspots Covered
- ✅ CUDA warmup before timing (avoids inflated first-run latency)
- ✅ `torch.cuda.synchronize()` for accurate GPU timing
- ✅ Audio resampling to 16kHz mono (Parakeet requirement)
- ✅ Chunk overlap to avoid word splitting at boundaries
- ✅ Min chunk length guard (< 0.1s chunks can crash)
- ✅ `model.freeze()` to disable gradients (lower memory, faster inference)
- ✅ GPU memory monitoring
- ✅ Latency percentiles (p50/p95/p99), not just averages
- ✅ RTF (Real-Time Factor) and RTFx computation
- ✅ Temp file cleanup after each chunk